# Análisis de Incidencias en Rodalies Renfe
## 1. Descripción del Proyecto

Este cuaderno tiene como objetivo explorar, limpiar y analizar el conjunto de datos de incidencias en la red de Rodalies Renfe. A través de este análisis se busca identificar patrones temporales, las líneas y tramos más afectados, la distribución del tipo de incidencias y el impacto en los tiempos de retraso de los usuarios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import re

# Configuració de la visualització
pd.set_option('display.max_columns', None)
sns.set_theme(style="darkgrid")

# Carga del Excel vigente compartido con los otros analisis
import sys
PROJECT_ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "src/analysis/rmove_general.py").is_file())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.rmove_general import cargar_catalogo
INCIDENCIAS_PATH = PROJECT_ROOT / "data/raw/rodalies/RMOVE_Incidencias_Barcelona_Jul2024_Ago2025_v07.xlsx"
df = cargar_catalogo(INCIDENCIAS_PATH)

# Inspecció bàsica de volum i primers registres
print(f"Dimensions del dataset: {df.shape[0]} files x {df.shape[1]} columnes\n")
df.head()

### Preprocesamiento de los datos
Fechas transformadas a formato dd/mm/yyyy, definición del orden de la gravedad de la incidencia y extracción del tiempo de retraso en formato numérico

In [ ]:
# 2. Conversión a formato datetime
# La fecha ya esta normalizada por cargar_catalogo.

# 3. Conversión de gravedad a Categoría Ordinal
orden_gravedad = ['Baja', 'Media', 'Alta', 'Muy alta']
df['severity'] = pd.Categorical(df['severity'], categories=orden_gravedad, ordered=True)

# 4. Extracción de minutos numéricos a partir del texto
def extraer_minutos(val):
    if pd.isna(val) or str(val).strip().lower() in ['no especifica', 'no especificada']:
        return np.nan
    val_str = str(val).lower()
    # Conversión de horas aisladas (ej. 4h, 12h)
    if 'h' in val_str and 'min' not in val_str:
        match = re.search(r'(\d+)', val_str)
        if match:
            return float(match.group(1)) * 60
    # Extracción de números en rangos o valores directos
    numeros = [float(n) for n in re.findall(r'\d+', val_str)]
    return np.mean(numeros) if numeros else np.nan

df['minutos_retraso_num'] = df['dalay_minutes'].apply(extraer_minutos)

df.info()

### Clasificación 2 — Desagregación por Líneas Individuales
Desglosamiento de la afectación en múltiples líneas y ordenadas por numero de incidencias


In [ ]:
# Reemplazar delimitadores y separar líneas en listas
df_lineas = df.copy()
df_lineas['linea_split'] = df_lineas['line'].str.replace(';', ',').str.split(',')

# Expandir cada línea a una fila independiente
df_exploded = df_lineas.explode('linea_split')
df_exploded['linea_limpia'] = df_exploded['linea_split'].str.strip()

# Resumen de incidencias por línea de tren
resumen_lineas = df_exploded.groupby('linea_limpia').agg(
    total_incidencias=('incident_id', 'count'),
    incidencias_altas=('severity', lambda x: (x.isin(['Alta', 'Muy alta'])).sum()),
    promedio_retraso=('minutos_retraso_num', 'mean'),
    retraso_total=('minutos_retraso_num', 'sum')
).reset_index().sort_values(by='total_incidencias', ascending=False).reset_index(drop=True)

resumen_lineas.head(15)

### Clasificación 4 — Matriz Temporal (Día de la Semana vs. Gravedad)
Organiza los datos según la recurrencia temporal para determinar qué días de la semana sufren mayor perturbación en el servicio.

In [ ]:
# Definir orden lógico de días de la semana
dias_ordenados = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
df['week_day'] = pd.Categorical(df['week_day'], categories=dias_ordenados, ordered=True)

# Tabla cruzada: Día de la semana vs Gravedad
matriz_temporal = pd.crosstab(
    df['week_day'],
    df['severity'],
    margins=True,
    margins_name='Total'
)

matriz_temporal


In [ ]:

# Preparar los datos para los gráficos, sin la fila ni la columna de totales
matriz_grafico = matriz_temporal.drop(index='Total', columns='Total', errors='ignore')
colores_gravedad = {
    'Baja': '#4C9F70',
    'Media': '#E3B341',
    'Alta': '#E07A3F',
    'Muy alta': '#C94C4C'
}
colores = [colores_gravedad.get(columna, '#777777') for columna in matriz_grafico.columns]

# Gráfico 1: número de incidencias por día y gravedad
ax = matriz_grafico.plot(
    kind='bar',
    stacked=True,
    figsize=(11, 5),
    color=colores,
    edgecolor='white'
)
ax.set_title('Incidencias por día de la semana y gravedad')
ax.set_xlabel('Día de la semana')
ax.set_ylabel('Número de incidencias')
ax.legend(title='Gravedad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Gráfico 2: porcentaje de cada gravedad dentro de cada día
matriz_porcentajes = matriz_grafico.div(matriz_grafico.sum(axis=1).replace(0, np.nan), axis=0) * 100
ax = matriz_porcentajes.plot(
    kind='bar',
    stacked=True,
    figsize=(11, 5),
    color=colores,
    edgecolor='white'
)
ax.set_title('Distribución porcentual de la gravedad por día')
ax.set_xlabel('Día de la semana')
ax.set_ylabel('Porcentaje de incidencias (%)')
ax.set_ylim(0, 100)
ax.legend(title='Gravedad')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Clasificación por minutos de interrupción


In [ ]:
# Clasificar cada incidencia según el tiempo de retraso
rangos_retraso = [-np.inf, 15, 30, 60, 120, np.inf]
etiquetas_retraso = [
    'Hasta 15 min',
    'De 15 a 30 min',
    'De 30 a 60 min',
    'De 60 a 120 min',
    'Más de 120 min'
]

incidencias_por_tiempo = df.copy()
incidencias_por_tiempo['rango_retraso'] = pd.cut(
    incidencias_por_tiempo['minutos_retraso_num'],
    bins=rangos_retraso,
    labels=etiquetas_retraso
)
incidencias_por_tiempo['rango_retraso'] = (
    incidencias_por_tiempo['rango_retraso']
    .cat.add_categories('Sin dato')
    .fillna('Sin dato')
)

# Resumen de incidencias por rango
orden_rangos = etiquetas_retraso + ['Sin dato']
resumen_tiempos = (
    incidencias_por_tiempo['rango_retraso']
    .value_counts(sort=False)
    .reindex(orden_rangos, fill_value=0)
    .rename_axis('Rango de retraso')
    .reset_index(name='Incidencias')
)
resumen_tiempos['Porcentaje'] = (
    resumen_tiempos['Incidencias'] / len(incidencias_por_tiempo) * 100
).round(1)

resumen_tiempos


In [ ]:
# Gráfico: número de incidencias en cada rango de retraso
ax = resumen_tiempos.set_index('Rango de retraso')['Incidencias'].plot(
    kind='bar',
    figsize=(10, 5),
    color=['#4C9F70', '#9DBB55', '#E3B341', '#E07A3F', '#C94C4C', '#9A9A9A'],
    edgecolor='white'
)
ax.set_title('Incidencias por rango de tiempo de retraso')
ax.set_xlabel('Rango de retraso')
ax.set_ylabel('Número de incidencias')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()